# Helopeltis spp. Detection — YOLOv8-n Fine-tuning

**Workstream A · TINAT ML Contribution — Angel Bohorquez, ML Intern**

---

## Phase 1 — 1-class model (Helopeltis only)

**Dataset:** Roboflow `helopeltis-yz1xl-sljp8` — 483 annotated images, CC BY 4.0  
**Backbone:** YOLOv8-n pretrained on COCO (will be replaced by the handover IP102 checkpoint when available)  
**Goal:** working detector on Helopeltis that can be fine-tuned again when field photos arrive

| Split | Images | Instances |
|-------|--------|-----------|
| train | 386    | ~397      |
| val   | 48     | ~43       |
| test  | 49     | ~43       |

**When the handover weights arrive:** update `BASELINE_WEIGHTS` in §6 and change `nc` to 11 in §5.

## 1. Environment

In [ ]:
# Run once — skip if already installed
# !pip install -r ../requirements.txt

In [2]:
import sys
from pathlib import Path

import yaml
import torch
from ultralytics import YOLO

ROOT      = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_RAW  = ROOT / "data" / "raw"
MODELS    = ROOT / "models"
HELO_DIR  = DATA_RAW / "helopeltis"   # extracted Roboflow dataset lives here

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print(f"Dataset root : {HELO_DIR}")

PyTorch      : 2.12.0+cpu
CUDA         : False
Dataset root : C:\Users\AngelTatianaMendozaB\OneDrive - ITS Angelo Rizzoli\Desktop\G\Helopeltis_detection\data\raw\helopeltis


## 2. Data verification

The Roboflow export already contains `train / valid / test` splits.  
This cell counts images and label files to confirm nothing is missing.

In [3]:
from collections import Counter

total_imgs = total_labels = 0
instance_counts = Counter()

for split in ["train", "valid", "test"]:
    imgs   = list((HELO_DIR / split / "images").glob("*.*"))
    labels = list((HELO_DIR / split / "labels").glob("*.txt"))
    instances = sum(
        len([l for l in lf.read_text().strip().splitlines() if l])
        for lf in labels
    )
    print(f"{split:6s}: {len(imgs):4d} images | {len(labels):4d} labels | {instances:4d} instances")
    total_imgs   += len(imgs)
    total_labels += len(labels)
    instance_counts[split] = instances

print(f"\nTotal images    : {total_imgs}")
print(f"Total instances : {sum(instance_counts.values())}")
print("\nDataset looks good — minimum 200 instances met:", sum(instance_counts.values()) >= 200)

train :  386 images |  386 labels |  386 instances
valid :   48 images |   48 labels |   48 instances
test  :   49 images |   49 labels |   49 instances

Total images    : 483
Total instances : 483

Dataset looks good — minimum 200 instances met: True


## 3. Train / val / test split

Already provided by Roboflow (80 / 10 / 10 %). No action needed.

## 4. Augmentation

YOLOv8 applies augmentation automatically during training:  
mosaic, random flip, HSV shift, scale, translate.

Extra augmentation parameters passed at training time (§6):  
- `degrees=15` — small rotations (field photos vary in angle)  
- `hsv_v=0.5` — brightness variation (outdoor lighting)  
- `mosaic=1.0` — combines 4 images per tile (improves small-object detection)

When field photos arrive they can be added to `train/images` and `train/labels`  
and fine-tuning can resume from the checkpoint produced here.

## 5. Dataset YAML

**Phase 1:** 1 class (`Helopeltis spp.`).  
When the handover weights arrive, expand `nc` to 11 and add the 10 IP102 class names.

In [4]:
dataset_yaml = {
    "path": str(HELO_DIR),        # absolute path — YOLOv8 resolves train/val/test relative to this
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    # Phase 1: single class.
    # Phase 2 (with the handover weights): nc=11, add IP102 names before 'Helopeltis spp.'
    "nc":    1,
    "names": ["Helopeltis spp."],
}

yaml_path = HELO_DIR / "helopeltis_phase1.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True, sort_keys=False)

print(f"YAML written → {yaml_path}")
print(yaml.dump(dataset_yaml, sort_keys=False))

YAML written → C:\Users\AngelTatianaMendozaB\OneDrive - ITS Angelo Rizzoli\Desktop\G\Helopeltis_detection\data\raw\helopeltis\helopeltis_phase1.yaml
path: C:\Users\AngelTatianaMendozaB\OneDrive - ITS Angelo Rizzoli\Desktop\G\Helopeltis_detection\data\raw\helopeltis
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- Helopeltis spp.



## 6. Training

**Phase 1:** start from `yolov8n.pt` (COCO pretrained — publicly available).  
**Phase 2 (future):** replace `BASELINE_WEIGHTS` with the handover IP102 checkpoint  
and run fine-tuning from the best weights produced here.

In [5]:
# Phase 1 — COCO pretrained base (downloads automatically on first run)
BASELINE_WEIGHTS = "yolov8n.pt"
# Phase 2 — uncomment when the handover checkpoint is available:
# BASELINE_WEIGHTS = str(MODELS / "helopeltis_yolov8n_baseline.pt")

model = YOLO(BASELINE_WEIGHTS)

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=1e-3,          # standard LR for phase 1 (no prior knowledge to preserve)
    patience=10,       # early stopping
    degrees=15,        # rotation augmentation — field photos vary in angle
    hsv_v=0.5,         # brightness variation — outdoor lighting conditions
    mosaic=1.0,
    project=str(MODELS),
    name="helopeltis_phase1",
    exist_ok=True,
)

print(f"\nBest weights → {MODELS / 'helopeltis_phase1' / 'weights' / 'best.pt'}")

New https://pypi.org/project/ultralytics/8.4.83 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.12.10 torch-2.12.0+cpu CPU (AMD Ryzen 7 7730U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\AngelTatianaMendozaB\OneDrive - ITS Angelo Rizzoli\Desktop\G\Helopeltis_detection\data\raw\helopeltis\helopeltis_phase1.yaml, degrees=15, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\AngelTatianaMendozaB\\OneDrive - ITS Angelo Rizzoli\\Desktop\\G\\Helopeltis_detection\\models\\helopeltis_phase1\\weights\\last.pt'

## 7. Evaluation

Key metrics for Phase 1:  
- **mAP@0.5** — target ≥ 0.70 (lower bar than phase 2 — single class, internet photos)  
- **Recall** — target ≥ 0.75 — false negatives (missed pests) are costly in the field  
- **Confusion matrix** — check for false positives on leaf background

In [ ]:
best_weights = MODELS / "helopeltis_phase1" / "weights" / "best.pt"
model = YOLO(str(best_weights))
metrics = model.val(data=str(yaml_path), split="test")

print(f"mAP@0.5          : {metrics.box.map50:.3f}")
print(f"Helopeltis P     : {metrics.box.p[0]:.3f}")
print(f"Helopeltis R     : {metrics.box.r[0]:.3f}")
print(f"mAP@0.5:0.95     : {metrics.box.map:.3f}")

## 8. Export — ONNX + TFLite

Ambos formatos exportados exitosamente:
- **ONNX** → `models/helopeltis_phase1.onnx` — usado por `fusion.py` `HelopeltisDetector` en servidor
- **TFLite** → `models/helopeltis_phase1.tflite` — listo para deployment mobile (Flutter / FastAPI CPU)

In [ ]:
import shutil

# Copiar ONNX a models/
onnx_src  = best_weights.parent / "best.onnx"
onnx_dest = MODELS / "helopeltis_phase1.onnx"
if not onnx_dest.exists():
    shutil.copy(onnx_src, onnx_dest)
print(f"ONNX   : {onnx_dest}  ({onnx_dest.stat().st_size / 1e6:.1f} MB)")

# Copiar TFLite a models/
tflite_src  = best_weights.parent / "best_saved_model" / "best_float32.tflite"
tflite_dest = MODELS / "helopeltis_phase1.tflite"
if not tflite_dest.exists():
    shutil.copy(tflite_src, tflite_dest)
print(f"TFLite : {tflite_dest}  ({tflite_dest.stat().st_size / 1e6:.1f} MB)")

print("\nWorkstream A completo. fusion.py HelopeltisDetector usa el ONNX automaticamente.")

## Next steps

**After Phase 1 training:**
1. Check mAP@0.5 and recall — if recall < 0.70 consider adding GBIF images
2. Wire `models/helopeltis_phase1.tflite` into `fusion.py` for demo inference

**When field photos arrive:**
1. Add images → `data/raw/helopeltis/train/images/` + labels → `data/raw/helopeltis/train/labels/`
2. Re-run §6 with `BASELINE_WEIGHTS = str(MODELS / 'helopeltis_phase1/weights/best.pt')` and lower `lr0=1e-4`

**When the handover weights arrive:**
1. Update `BASELINE_WEIGHTS` in §6 to the handover `.pt` file
2. Change `nc=11` and expand `names` in §5 (add 10 IP102 classes before `Helopeltis spp.`)
3. Re-label all Helopeltis annotations with class index `10` instead of `0`